In [1]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 11.2 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from itertools import combinations
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/heartdisease/Heart_Disease_Prediction.csv
/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

CONFIG = config()

In [4]:
train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

org = pd.read_csv('/kaggle/input/heartdisease/Heart_Disease_Prediction.csv')
org['source'] = 'original'


combine = pd.concat([train.drop(columns='id'), test.drop(columns='id')], ignore_index=True).reset_index()



In [5]:
NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if combine[col].nunique() > 40]

BINS = []
for q in [5]:
    for c in HIGH_CARDINALITY:
        n = f'{c}_{q}_bin'
        train_bins, bins = pd.qcut(train[c], q=q, labels=False, retbins=True, duplicates='drop')
        train[n] = train_bins
        test[n] = pd.cut(test[c], bins=bins, labels=False, include_lowest=True)
        BINS.append(n)

print(BINS)
print('='*30)
print(len(BINS))

['Age_5_bin', 'BP_5_bin', 'Cholesterol_5_bin', 'Max HR_5_bin', 'ST depression_5_bin']
5


In [6]:
for df, name in zip([train, test, org], ['train', 'test', 'original']):
    print(f'NULL VALUE COUNTS FOR {name}:')
    print(df.isnull().sum())
    print('='*30)
    print(f'{name} shape:')
    print(df.shape)
    print('='*30)
    if name == 'train':
        print('General EDA -- TRAIN ONLY', end='\n')
        print(f'Dtypes :', end='\n')
        print(df.dtypes)
        print('='*30)
        print(f'NUMBER OF UNIQUE VALUES :', end='\n')
        print(df.nunique())
        print('='*30)
    

NULL VALUE COUNTS FOR train:
id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
source                     0
Age_5_bin                  0
BP_5_bin                   0
Cholesterol_5_bin          0
Max HR_5_bin               0
ST depression_5_bin        0
dtype: int64
train shape:
(630000, 21)
General EDA -- TRAIN ONLY
Dtypes :
id                           int64
Age                          int64
Sex                          int64
Chest pain type              int64
BP                           int64
Cholesterol                  int64
FBS over 120                 int64
EKG results                  int64
Max HR  

In [7]:
print(NUMS)
print(HIGH_CARDINALITY)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']


In [8]:
# CATS = []
# for c in NUMS:
#     n = f'{c}_cat'
#     combine[n] = combine[c].astype(str).astype('category')
#     CATS.append(n)

# print(CATS)
# print('='*30)
# print(len(CATS))

In [9]:
# INTER = []
# for c1, c2 in combinations(CATS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = (combine[c1].astype(str) + '_' + combine[c2].astype(str)).astype('category')
#     INTER.append(n)

# print(INTER)
# print('='*30)
# print(len(INTER))

In [10]:
# ENC = []

# for c in INTER:
#     n = f'{c}_enc'
#     combine[n], _ = pd.factorize(combine[c])
#     ENC.append(n)

# print(ENC)
# print('='*30)
# print(len(ENC))

In [11]:
train = combine.loc[combine['source']=='train']
test = combine.loc[combine['source']=='test']
org = combine.loc[combine['source']=='original']

In [12]:
class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

FEATURES = [col for col in combine.columns if col not in ['id', 'Heart Disease', 'source', 'index']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET].map(class_mapping)

X_test = test[FEATURES]

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
13


In [13]:
xgb_params = {
    'n_estimators': 10000,
    'learning_rate': 0.005,
    'subsample': 0.8,
    # 'colsample_by_tree': 0.7,
    # 'sampling_method': 'gradient_based',
    # 'reg_alpha': 2.0,
    # 'reg_lambda': 4.0,
    'eval_metric': 'auc',
    'enable_categorical': True,
    'tree_method': 'hist',
    'device': 'cuda',
    'early_stopping_rounds': 100,
    'random_state': CONFIG.SEED
}

lgb_params = {
    'n_estimators': 10_000,
    'learning_rate': 0.005,
    'max_depth': 8,                 # same philosophy
    'num_leaves': 2 ** 8,           # typical rule: ≤ 2^max_depth
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    # 'reg_lambda': 4.0,
    'random_state': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'auc',
    'n_jobs': -1,
    # 'verbose': 200,
    'verbosity':-1,
    # 'device': 'cuda'
    # 'cat_feature': CATS,            # pass categorical indices/names
}

cat_params = {
    'iterations': 10_000,          # same as n_estimators
    'learning_rate': 0.005,
    'depth': 8,                    # max_depth equivalent
    'subsample': 0.8,              # bagging
    'colsample_bylevel': 0.7,      # feature fraction per split
    'reg_lambda': 4.0,             # L2
    'random_seed': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'AUC',
    'thread_count': -1,            # use all cores
    'verbose': 200,                # same as XGB verbose
    # 'verbosity':-1
    # 'cat_features': CATS,          # list of column names / indices
}

real_mlp_params = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'val_metric_name': '1-auc_ovo',
        'n_epochs': 60,
        'batch_size': 1024,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.01,
        'ls_eps': 0.0,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 3,
        'p_drop': 0.1,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }

In [14]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 0):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    # X_org_n = X_org.copy()
    # y_org_n = y_org.copy()
    # X_org_n = pd.concat([X_org_n]*20, axis=0, ignore_index=True)
    # y_org_n = pd.concat([y_org_n]*20, axis=0, ignore_index=True)
    # X_train = pd.concat([X_train, X_org_n], axis=0, ignore_index=True)
    # y_train = pd.concat([y_train, y_org_n], axis=0, ignore_index=True)
    # model = clone(xgb.XGBClassifier(**xgb_params))
    # model = clone(cb.CatBoostClassifier(**cat_params))
    # model = clone(lgb.LGBMClassifier(**lgb_params))

    # model = clone(RealMLP_TD_Classifier(**real_mlp_params))
    param_grid = {'colsample_bytree': 0.2364,
                  'gamma': 0.034283,
                  'max_depth': 6,
                  'reg_alpha': 0.71367,
                  'reg_lambda': 4.43564,
                  'subsample': 0.59394}

    model = xgb.XGBClassifier(**param_grid,
                          n_estimators=10000,
                          objective='binary:logistic',
                          eval_metric='auc',
                          learning_rate=0.01,
                          early_stopping_rounds=500,
                          max_bin=1024,
                          random_state=42,
                          enable_categorical=True,
                          device='cuda',
                          n_jobs=-1)
    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=500)
    preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = preds
    test_preds += (model.predict_proba(X_test)[:, 1] / CONFIG.N_FOLDS)
    print(f'SCORE FOR FOLD{i} : {roc_auc_score(y_val, preds)}')
    
overall_roc_auc = roc_auc_score(y, oof_preds)
print(f'SCORE ACROSS ALL FOLDS : {overall_roc_auc}')

[0]	validation_0-auc:0.84685
[500]	validation_0-auc:0.95451
[1000]	validation_0-auc:0.95528
[1500]	validation_0-auc:0.95563
[2000]	validation_0-auc:0.95579
[2500]	validation_0-auc:0.95587
[3000]	validation_0-auc:0.95591
[3500]	validation_0-auc:0.95592
[4000]	validation_0-auc:0.95593
[4500]	validation_0-auc:0.95592
[4717]	validation_0-auc:0.95592
SCORE FOR FOLD0 : 0.9559279083199679
[0]	validation_0-auc:0.84639
[500]	validation_0-auc:0.95345
[1000]	validation_0-auc:0.95422
[1500]	validation_0-auc:0.95455
[2000]	validation_0-auc:0.95469
[2500]	validation_0-auc:0.95476
[3000]	validation_0-auc:0.95480
[3500]	validation_0-auc:0.95481
[4000]	validation_0-auc:0.95482
[4500]	validation_0-auc:0.95482
[4862]	validation_0-auc:0.95481
SCORE FOR FOLD1 : 0.9548185637263029
[0]	validation_0-auc:0.84645
[500]	validation_0-auc:0.95441
[1000]	validation_0-auc:0.95510
[1500]	validation_0-auc:0.95539
[2000]	validation_0-auc:0.95553
[2500]	validation_0-auc:0.95560
[3000]	validation_0-auc:0.95563
[3500]	val

In [15]:
sample_sub['Heart Disease'] = test_preds
sample_sub.to_csv(f'submission_{overall_roc_auc}.csv', index=False)